# Silver Layer — ERP: Customer Location (loc_a101)

Transforms the bronze `workspace.bronze.loc_a101` table into `workspace.silver.customer_info2` with meaningful, readable column names and cleaned data.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (`CID` -> `customer_id`, `CNTRY` -> `country`) |
| 2 | **Clean Customer ID** | Trimmed whitespace and removed dash from `customer_id` (e.g., `AW-00011000` -> `AW00011000`) |
| 3 | **Trim Whitespace** | Trimmed leading/trailing whitespace from all string columns |
| 3 | **Country Standardization** | Unified country abbreviations with full names: `USA`/`US` -> `United States`, `DE` -> `Germany` |
| 4 | **NULL Country Handling** | Replaced NULL and blank country values with `Unknown` |

---

## Column Mapping (Bronze to Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `CID` | `customer_id` | Renamed + Trimmed + Dash removed |
| `CNTRY` | `country` | Renamed + Trimmed + Standardized |

---

## Country Value Mapping

| Bronze Value(s) | Silver Value | Count |
|---|---|---|
| `United States` | `United States` | 3,391 |
| `USA` | `United States` | 2,591 |
| `US` | `United States` | 1,500 |
| `Germany` | `Germany` | 1,214 |
| `DE` | `Germany` | 566 |
| `Australia` | `Australia` | 3,591 |
| `United Kingdom` | `United Kingdom` | 1,913 |
| `France` | `France` | 1,810 |
| `Canada` | `Canada` | 1,571 |
| `NULL`, ` ` (spaces) | `Unknown` | 337 |

---

## Implementation

All transformations are implemented in **Spark SQL**:
- `TRIM()` — remove leading/trailing whitespace
- `REGEXP_REPLACE()` — remove the dash from `customer_id`
- `CASE WHEN` — conditional mapping for country standardization
- `CREATE OR REPLACE TABLE ... AS SELECT` — write to silver layer

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.customer_info2 AS
SELECT
    REGEXP_REPLACE(TRIM(CID), '-', '')  AS customer_id,
    CASE
        WHEN TRIM(CNTRY) IN ('USA', 'US') THEN 'United States'
        WHEN TRIM(CNTRY) = 'DE'          THEN 'Germany'
        WHEN TRIM(CNTRY) = ''            THEN 'Unknown'
        WHEN CNTRY IS NULL               THEN 'Unknown'
        ELSE TRIM(CNTRY)
    END                                 AS country
FROM workspace.bronze.loc_a101

In [0]:
%sql
SELECT * FROM workspace.silver.customer_info2 LIMIT 10